In [ ]:
# MOSS-TTS against the same nine speakers, sentences, seed and reference clips.
# transformers is pinned because MOSS's remote code calls generation internals
# that 5.1 removed; upstream's card asks for 5.0.0 for the same reason.
!pip -q install "transformers>=5.0,<5.1" accelerate speechbrain faster-whisper jiwer pyloudnorm noisereduce num2words silero-vad onnxruntime


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import torch
import transformers

INPUT = Path("/kaggle/input")
print("mounted     :", sorted(p.name for p in INPUT.iterdir()) if INPUT.exists() else "nothing")

# The dataset mounts at /kaggle/input/datasets/<owner>/<slug>/, so the repo is
# found by a file that has to be inside it rather than by an assumed depth.
found = sorted(INPUT.rglob("repo/scripts/benchmark.py"))
if not found:
    for path in sorted(p for p in INPUT.rglob("*") if p.is_dir())[:40]:
        print("   ", path)
    raise SystemExit("no repo under /kaggle/input - see the directories above")
SOURCE = found[0].parent.parent
print("source      :", SOURCE)

REPO = Path("/kaggle/working/repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(SOURCE, REPO)

version = tuple(int(part) for part in transformers.__version__.split(".")[:2])
print("transformers:", transformers.__version__)
print("torch       :", torch.__version__)
print("gpus        :", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("bf16        :", torch.cuda.is_bf16_supported())

assert (5, 0) <= version < (5, 1), f"MOSS needs transformers 5.0.x, found {transformers.__version__}"
assert torch.cuda.device_count() >= 1, "no GPU - set the accelerator"


def run(*arguments, timeout=None):
    """Run the benchmark and return (exit code, wall-clock seconds)."""
    command = [sys.executable, "scripts/benchmark.py", *arguments]
    print(" ".join(command), flush=True)
    start = time.time()
    try:
        outcome = subprocess.run(command, cwd=REPO, text=True, timeout=timeout)
        code = outcome.returncode
    except subprocess.TimeoutExpired:
        print(f"TIMED OUT after {timeout}s - called off", flush=True)
        code = -1
    elapsed = time.time() - start
    print(f"exit {code} in {elapsed / 60:.1f} min", flush=True)
    return code, elapsed


In [ ]:
# Same flags as every other engine apart from the device, so the cells line up
# with every earlier run. Bounded: the single-cell timing run predicts about
# 1.2 h, and this stops at two.
code, elapsed = run("--label", "moss", "--engine", "moss", "--device", "cuda",
                    timeout=7200)
print(f"render wall clock: {elapsed / 60:.1f} min")

# benchmark.py records failures in its results file, not in its exit code: an
# earlier run exited 0 with all 27 cells failed and the kernel said COMPLETE.
path = REPO / "eval/results/moss.json"
if path.is_file():
    summary = json.loads(path.read_text())["summary"]
    print("moss summary:", summary, flush=True)
    print("failures:", summary.get("failures"), " renders:", summary.get("renders"))
else:
    print("no results file - the render did not get far enough to write one")


In [ ]:
# The scale check, after the measurement is safely on disk.
#
# Every number on record was produced with transformers 5.14; this kernel runs
# 5.0.x, so its embedder and recogniser are a different build. Re-score renders
# whose answer is already established and compare. This reports rather than
# asserts, because stopping here would skip collecting what the render made.
code, elapsed = run("--label", "baseline-moss-env", "--score-only", "--replay-from", "baseline",
                    "--device", "cuda", timeout=1800)
path = REPO / "eval/results/baseline-moss-env.json"
if path.is_file():
    value = json.loads(path.read_text())["summary"]["held_out_mean"]
    drift = value - 0.5196
    print(f"control: {value:.4f} vs 0.5196 known ({drift:+.4f})")
    print("ON the established scale" if abs(drift) < 0.0005 else "OFF THE ESTABLISHED SCALE")
else:
    print("control did not produce a results file")


In [ ]:
destination = Path("/kaggle/working/results")
destination.mkdir(exist_ok=True)
for path in (REPO / "eval/results").glob("*.json"):
    shutil.copy2(path, destination / path.name)

renders = Path("/kaggle/working/moss_renders")
renders.mkdir(exist_ok=True)
for path in (REPO / "outputs/benchmark/moss").glob("*.wav"):
    shutil.copy2(path, renders / path.name)
print("renders saved:", len(list(renders.glob("*.wav"))))

for name in ("moss", "baseline-moss-env"):
    path = destination / f"{name}.json"
    if path.is_file():
        data = json.loads(path.read_text())["summary"]
        print(name, {k: data.get(k) for k in ("held_out_mean", "cer_mean", "renders", "failures")})
